In [1]:
import pandas as pd
import numpy as np
import os
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")



In [2]:
# Read the primary contacts file
file_path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Payment\Contacts_Payment\Contacts.csv"
df_file = pd.read_csv(file_path, low_memory=False)

In [3]:
df_file[df_file['session_dt'].isna()]
print(df_file.columns)

Index(['year1', 'month1', 'wk1', 'session_dt', 'marketplace_id',
       'is_shopsy_order', 'LOB', 'issue_type', 'sub_issue_type',
       'sub_sub_issue_type', 'ssi', 'merchant_id', 'live_response_code',
       'transaction_status', 'merchant_status', 'pg_id', 'transaction_source',
       'payment_instrument', 'FRM_OCR_Flag', 'flipkart_emi_flag',
       'marketplace_context', 'Minutes_flag', 'emi_flag', 'contacts',
       'inc_cnt', 'odr_cnt', 'acct_cnt'],
      dtype='object')


In [4]:
df_file=df_file[df_file['session_dt']!="session_dt"]
df_file=df_file[df_file['contacts']!="contacts"]
df_file=df_file[df_file['marketplace_id']!="marketplace_id"]
df_file['session_dt']=df_file['session_dt'].astype("datetime64[ns]")
df_file['contacts'] = pd.to_numeric(df_file['contacts'], errors = 'coerce')
df_file['FRM_OCR_Flag']=df_file['FRM_OCR_Flag'].astype(str).str.lower()
df_file['is_shopsy_order']=df_file['is_shopsy_order'].astype(str).str.lower()
#df_file['Minutes_flag'] = df_file['Minutes_flag'].astype(str).str.lower()
df_file=df_file.sort_values(by=['session_dt'])
jn_path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\JN_mapping.xlsx"
df_jn = pd.read_excel(jn_path)
merged_df = pd.merge(df_file, df_jn, how="left", left_on="ssi", right_on="current_iss")

##print(merged_df)
print(merged_df.dtypes)
print(merged_df['FRM_OCR_Flag'].unique())
print(merged_df['is_shopsy_order'].unique())
print(merged_df['Minutes_flag'].unique())
print(merged_df['marketplace_id'].unique())

year1                           int64
month1                          int64
wk1                             int64
session_dt             datetime64[ns]
marketplace_id                 object
is_shopsy_order                object
LOB                            object
issue_type                     object
sub_issue_type                 object
sub_sub_issue_type             object
ssi                            object
merchant_id                    object
live_response_code             object
transaction_status             object
merchant_status                object
pg_id                          object
transaction_source             object
payment_instrument             object
FRM_OCR_Flag                   object
flipkart_emi_flag              object
marketplace_context            object
Minutes_flag                   object
emi_flag                       object
contacts                        int64
inc_cnt                         int64
odr_cnt                         int64
acct_cnt    

In [5]:
jn_path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\JN_mapping.xlsx"
df_jn = pd.read_excel(jn_path)
merged_df = pd.merge(df_file, df_jn, how="left", left_on="ssi", right_on="current_iss")

In [6]:
conversion_path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Conversation_Factors.xlsx"
df_conversion = pd.read_excel(conversion_path)

In [7]:
   
filtered_data_total = merged_df[
    (~merged_df["marketplace_id"].isin(["HYPERLOCAL"])) &
    (~merged_df["FRM_OCR_Flag"].isin(["true"])) &
     (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) 
].copy()

In [8]:
total_contacts_full = (filtered_data_total
                      .groupby("session_dt")["contacts"]
                      .sum()
                      .reset_index()
                       .rename(columns={"contacts": "total_contacts_full"})
                        )
##last_5 = total_contacts_full.sort_values("session_dt").tail(5)

In [9]:
print(total_contacts_full.tail(10))

   session_dt  total_contacts_full
12 2025-12-13               200629
13 2025-12-14               186461
14 2025-12-15               205307
15 2025-12-16               209897
16 2025-12-17               211252
17 2025-12-18               203733
18 2025-12-19               198332
19 2025-12-20               197137
20 2025-12-21               183495
21 2025-12-22                 3014


In [10]:
merged_df["live_response_code_flag"] = merged_df["live_response_code"].apply(
    lambda x: 'SUCCESS' if x == 'SUCCESS' else 'Not_SUCCESS'
)

filtered_data = merged_df[
    (merged_df["Minutes_flag"].isin(["Minutes"])) &
    (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
    (merged_df["JN"] == "PAYMENTS") & 
       #(merged_df["merchant_id"] == 'mp_flipkart') &
    (merged_df["ssi"].isin([
        "Order processing Related->Online Payment Issues->Amount Debited Order not confirmed",
        "Product Purchase Queries->Payment related Queries->NA"
    ]))
].copy()


filtered_data["contacts"] = pd.to_numeric(filtered_data["contacts"], errors='coerce')
group_cols = ["session_dt"]

overall_transaction_data = (
    filtered_data.groupby(group_cols)["contacts"].sum().reset_index().rename(columns={"contacts": "overall_transaction_count"})
)

successful_transaction = (
    filtered_data[filtered_data["live_response_code_flag"] == "SUCCESS"]
    .groupby(group_cols)["contacts"].sum().reset_index().rename(columns={"contacts": "successful_count"})
)

failed_data = (
    filtered_data[
        (filtered_data["live_response_code_flag"] == "Not_SUCCESS") &
        (filtered_data["transaction_status"] == "FAILED")
    ]
    .groupby(group_cols)["contacts"].sum().reset_index().rename(columns={"contacts": "failed_count"})
)

order_confirmed_data = (
    filtered_data[
        (filtered_data["live_response_code_flag"] == "Not_SUCCESS") &
        (filtered_data["transaction_status"] == "SUCCESS") &
        (filtered_data["merchant_status"] == "CAPTURED")
    ]
    .groupby(group_cols)["contacts"].sum().reset_index().rename(columns={"contacts": "order_confirmed_count"})
)

transaction_failed_data = (
    filtered_data[
        (filtered_data["live_response_code_flag"] == "Not_SUCCESS") &
        (filtered_data["transaction_status"] == "SUCCESS") &
        (filtered_data["merchant_status"] == "INVALIDATED")
    ]
    .groupby(group_cols)["contacts"].sum().reset_index().rename(columns={"contacts": "transaction_failed_count"})
)


summary_frames = [
    overall_transaction_data.set_index("session_dt").rename(columns={"overall_transaction_count": "Total_transaction"}),
    successful_transaction.set_index("session_dt").rename(columns={"successful_count": "Successful_transaction"}),
    failed_data.set_index("session_dt").rename(columns={"failed_count": "Failed_transaction"}),
    order_confirmed_data.set_index("session_dt").rename(columns={"order_confirmed_count": "Order_confirmed_transaction"}),
    transaction_failed_data.set_index("session_dt").rename(columns={"transaction_failed_count": "Transaction_failed_transaction"})
]

summary_df = pd.concat(summary_frames, axis=1).fillna(0)

# Step 5: Transpose for final report layout
final_summary = summary_df.T
final_summary.columns = pd.to_datetime(final_summary.columns, errors='coerce').strftime('%d-%m-%Y')
final_summary.reset_index(inplace=True)
final_summary.rename(columns={"index": "Transaction_Data"}, inplace=True)


df_conversion["session_dt"] = pd.to_datetime(df_conversion["Date"], format="%d-%m-%Y", errors="coerce")
conversion_df = pd.merge(df_conversion, total_contacts_full, on="session_dt", how="inner")
#conversion_df["conversion_pct"] = (conversion_df["Convertion_factor"] / conversion_df["total_contacts_full"]).round(2)
conversion_df["conversion_pct"] = (conversion_df["Convertion_factor"] / conversion_df["total_contacts_full"])
conversion_df["date_str"] = conversion_df["session_dt"].dt.strftime('%d-%m-%Y')
conversion_map = dict(zip(conversion_df["date_str"], conversion_df["conversion_pct"]))


for col in final_summary.columns[1:]:
    if col in conversion_map:
        final_summary[col] = final_summary[col] * conversion_map[col]

# Final rounding
final_summary.iloc[:, 1:] = final_summary.iloc[:, 1:].fillna(0).round(0).astype(int)

#print(final_summary)

###-------------------------##
##Adonc------##


merged_df["live_response_code_flag"] = merged_df["live_response_code"].apply(
    lambda x: 'SUCCESS' if x == 'SUCCESS' else 'Not_SUCCESS'
)

filtered_data9 = merged_df[
    (merged_df["Minutes_flag"].isin(["Minutes"])) &
    (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
    (merged_df["JN"] == "PAYMENTS") & 
   # (merged_df["payment_instrument"] != 'COD') &
    #(merged_df["merchant_id"] == 'mp_flipkart') &
    (merged_df["ssi"].isin([
        "Order processing Related->Online Payment Issues->Amount Debited Order not confirmed"
    ]))
].copy()

filtered_data9["contacts"] = pd.to_numeric(filtered_data9["contacts"], errors='coerce')
group_cols = ["session_dt"]


overall_transaction_data1 = (
    filtered_data9.groupby(group_cols)["contacts"].sum().reset_index().rename(columns={"contacts": "overall_transaction_count1"})
)

successful_transaction1 = (
    filtered_data9[filtered_data9["live_response_code_flag"] == "SUCCESS"]
    .groupby(group_cols)["contacts"].sum().reset_index().rename(columns={"contacts": "successful_count"})
)

failed_data1 = (
    filtered_data9[
        (filtered_data9["live_response_code_flag"] == "Not_SUCCESS") &
        (filtered_data9["transaction_status"] == "FAILED")
    ]
    .groupby(group_cols)["contacts"].sum().reset_index().rename(columns={"contacts": "failed_count"})
)

order_confirmed_data1 = (
    filtered_data9[
        (filtered_data9["live_response_code_flag"] == "Not_SUCCESS") &
        (filtered_data9["transaction_status"] == "SUCCESS") &
        (filtered_data9["merchant_status"] == "CAPTURED")
    ]
    .groupby(group_cols)["contacts"].sum().reset_index().rename(columns={"contacts": "order_confirmed_count"})
)

transaction_failed_data1 = (
    filtered_data9[
        (filtered_data9["live_response_code_flag"] == "Not_SUCCESS") &
        (filtered_data9["transaction_status"] == "SUCCESS") &
        (filtered_data9["merchant_status"] == "INVALIDATED")
    ]
    .groupby(group_cols)["contacts"].sum().reset_index().rename(columns={"contacts": "transaction_failed_count"})
)


summary_frames = [
    overall_transaction_data1.set_index("session_dt").rename(columns={"overall_transaction_count": "Total_transaction"}),
    successful_transaction1.set_index("session_dt").rename(columns={"successful_count": "Successful_transaction"}),
    failed_data1.set_index("session_dt").rename(columns={"failed_count": "Failed_transaction"}),
    order_confirmed_data1.set_index("session_dt").rename(columns={"order_confirmed_count": "Order_confirmed_transaction"}),
    transaction_failed_data1.set_index("session_dt").rename(columns={"transaction_failed_count": "Transaction_failed_transaction"})
]

summary_df1 = pd.concat(summary_frames, axis=1).fillna(0)

# Step 5: Transpose for final report layout
final_summary1 = summary_df1.T
final_summary1.columns = pd.to_datetime(final_summary1.columns, errors='coerce').strftime('%d-%m-%Y')
final_summary1.reset_index(inplace=True)
final_summary1.rename(columns={"index": "Transaction_Data"}, inplace=True)


df_conversion["session_dt"] = pd.to_datetime(df_conversion["Date"], format="%d-%m-%Y", errors="coerce")
conversion_df = pd.merge(df_conversion, total_contacts_full, on="session_dt", how="inner")
#conversion_df["conversion_pct"] = (conversion_df["Convertion_factor"] / conversion_df["total_contacts_full"]).round(2)
conversion_df["conversion_pct"] = (conversion_df["Convertion_factor"] / conversion_df["total_contacts_full"])
conversion_df["date_str"] = conversion_df["session_dt"].dt.strftime('%d-%m-%Y')
conversion_map = dict(zip(conversion_df["date_str"], conversion_df["conversion_pct"]))


for col in final_summary1.columns[1:]:
    if col in conversion_map:
        final_summary1[col] = final_summary1[col] * conversion_map[col]

# Final rounding
final_summary1.iloc[:, 1:] = final_summary1.iloc[:, 1:].fillna(0).round(0).astype(int)

#print(final_summary)


###-----------------------##
###Payment_Tranaction_Source
filtered_data12= merged_df[(merged_df["Minutes_flag"].isin(["Minutes"])) &
                          (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
                           (merged_df["JN"]=="PAYMENTS")].copy()

transaction_order1 = ["AndroidApp", "iOSApp", "MobileSite", "ucweb", "WEB"]
filtered_data12["transaction_source"] =pd.Categorical(filtered_data12["transaction_source"], categories=transaction_order1, ordered=True)

transaction_source_summary = (filtered_data12.groupby(["transaction_source","session_dt"], observed=False)["contacts"].sum().reset_index()
                              .rename(columns={"contacts": "transaction_source_count"}))
pivot_transaction_source_countact = transaction_source_summary.pivot_table(index="transaction_source",
                                                                     columns="session_dt",
                                                                     values = "transaction_source_count",
                                                                     aggfunc ='sum',
                                                                     observed =False).fillna(0)

pivot_transaction_source_countact.columns=pd.to_datetime(pivot_transaction_source_countact.columns, errors='coerce')
pivot_transaction_source_countact = pivot_transaction_source_countact.sort_index(axis=1)
pivot_transaction_source_countact.columns = pivot_transaction_source_countact.columns.strftime('%d-%m-%Y')
pivot_transaction_source_countact.reset_index(inplace=True)

for col in pivot_transaction_source_countact.columns[1:]:
    if col in conversion_map:
        pivot_transaction_source_countact[col] = pivot_transaction_source_countact[col] * conversion_map[col]

# Final rounding
pivot_transaction_source_countact.iloc[:, 1:] = pivot_transaction_source_countact.iloc[:, 1:].fillna(0).round(0).astype(int)


#print(pivot_transaction_source_countact.head(10))


##-------------------------------------##
#pivot_payment_instrument_countact###

filtered_data2= merged_df[(merged_df["Minutes_flag"].isin(["Minutes"])) &
                          (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
                          (merged_df["JN"]=="PAYMENTS")].copy()

emi_pg_ids =['cfa_juspay_dmi_emi',
    'cfa_juspay_fibe_emi',
    'cfa_juspay_abfl_emi',
    'cfa_juspay_chola_emi',
    'cfa_juspay_tvs_emi'
]

filtered_data2["payment_instrument"] = filtered_data2.apply(
lambda row: 'fk_emi' if row['pg_id'] in emi_pg_ids else row['payment_instrument'], axis=1)

filtered_data2["payment_instrument"] = filtered_data2["payment_instrument"].replace({"EGV":"EGV_wallet","EGV_WALLET":"EGV_wallet"})
transaction_order2 = ["UPI payment","PHONEPE","UPI_COLLECT","UPI_INTENT","FK_UPI","CREDIT","DEBIT","NET","COIN","EMI","EGV_wallet","WALLET","SUPERPAY_IN_INSTALLMENT","SUPERPAY_LATER","fk_emi"
]
filtered_data2["payment_instrument"] =pd.Categorical(filtered_data2["payment_instrument"], categories=transaction_order2, ordered=True)

payment_instrument_summary1 = (filtered_data2.groupby(["payment_instrument","session_dt"], observed=False)["contacts"].sum().reset_index()
                              .rename(columns={"contacts": "payment_instrument_count"}))

pivot_payment_instrument_countact = payment_instrument_summary1.pivot_table(index="payment_instrument",
                                                                     columns="session_dt",
                                                                     values = "payment_instrument_count",
                                                                     aggfunc ='sum',
                                                                     observed =False).fillna(0)

pivot_payment_instrument_countact.columns=pd.to_datetime(pivot_payment_instrument_countact.columns, errors='coerce')
pivot_payment_instrument_countact = pivot_payment_instrument_countact.sort_index(axis=1)
pivot_payment_instrument_countact.columns = pivot_payment_instrument_countact.columns.strftime('%d-%m-%Y')
pivot_payment_instrument_countact.reset_index(inplace=True)

for col in pivot_payment_instrument_countact.columns[1:]:
    if col in conversion_map:
        pivot_payment_instrument_countact[col] = pivot_payment_instrument_countact[col] * conversion_map[col]

# Final rounding
pivot_payment_instrument_countact.iloc[:, 1:] = pivot_payment_instrument_countact.iloc[:, 1:].fillna(0).round(0).astype(int)


#print(pivot_payment_instrument_countact)

##-------------------------------------##
#pivot_payment_Adonc_instrument_countact###

filtered_data6= merged_df[(merged_df["Minutes_flag"].isin(["Minutes"])) &
                          (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
                          (merged_df["JN"]=="PAYMENTS") &
                          (merged_df["ssi"]=="Order processing Related->Online Payment Issues->Amount Debited Order not confirmed")].copy()

print(filtered_data2['session_dt'].dtype)
print(filtered_data2['session_dt'].min())
emi_pg_ids =['cfa_juspay_dmi_emi',
    'cfa_juspay_fibe_emi',
    'cfa_juspay_abfl_emi',
    'cfa_juspay_chola_emi',
    'cfa_juspay_tvs_emi'
]

filtered_data6["payment_instrument"] = filtered_data6.apply(
lambda row: 'fk_emi' if row['pg_id'] in emi_pg_ids else row['payment_instrument'], axis=1)

filtered_data6["payment_instrument"] = filtered_data6["payment_instrument"].replace({"EGV":"EGV_wallet","EGV_WALLET":"EGV_wallet"})

transaction_order3 = ["PHONEPE","UPI_COLLECT","UPI_INTENT","FK_UPI","CREDIT","DEBIT","NET","COIN","EMI","EGV_wallet","WALLET","SUPERPAY_IN_INSTALLMENT","SUPERPAY_LATER","fk_emi"]
filtered_data6["payment_instrument"] =pd.Categorical(filtered_data6["payment_instrument"], categories=transaction_order3, ordered=True)

payment_instrument_Adonc_summary = (filtered_data6.groupby(["payment_instrument","session_dt"], observed=False)["contacts"].sum().reset_index()
                              .rename(columns={"contacts": "payment_instrument_Adonc_count"}))

pivot_payment_instrument_Adonc_countact = payment_instrument_Adonc_summary.pivot_table(index="payment_instrument",
                                                                     columns="session_dt",
                                                                     values = "payment_instrument_Adonc_count",
                                                                     aggfunc ='sum',
                                                                     observed =False).fillna(0)

pivot_payment_instrument_Adonc_countact.columns=pd.to_datetime(pivot_payment_instrument_Adonc_countact.columns, errors='coerce')
pivot_payment_instrument_Adonc_countact = pivot_payment_instrument_Adonc_countact.sort_index(axis=1)
pivot_payment_instrument_Adonc_countact.columns = pivot_payment_instrument_Adonc_countact.columns.strftime('%d-%m-%Y')
pivot_payment_instrument_Adonc_countact.reset_index(inplace=True)

for col in pivot_payment_instrument_Adonc_countact.columns[1:]:
    if col in conversion_map:
        pivot_payment_instrument_Adonc_countact[col] = pivot_payment_instrument_Adonc_countact[col] * conversion_map[col]

# Final rounding
pivot_payment_instrument_Adonc_countact.iloc[:, 1:] = pivot_payment_instrument_Adonc_countact.iloc[:, 1:].fillna(0).round(0).astype(int)


#print(pivot_payment_instrument_Adonc_countact.head(10))

####---------------------
### SSI View-------
merged_df["live_response_code_flag"] = merged_df["live_response_code"].apply(
    lambda x: 'SUCCESS' if x == 'SUCCESS' else 'Not_SUCCESS'
)

filtered_data11 = merged_df[
    (merged_df["Minutes_flag"].isin(["Minutes"])) &
    (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
     (merged_df["JN"] == "PAYMENTS")
 ].copy()

filtered_data11["contacts"] = pd.to_numeric(filtered_data11["contacts"], errors='coerce')
group_cols = ["session_dt"]

#Adonc_SSI_summary = (
#        filtered_data11.groupby(["session_dt","ssi"])["contacts"].sum().reset_index()
#        .rename(columns={"contacts":"Adonc_ssi_counts"})
#)

transaction_order7 = ['Order processing Related->Online Payment Issues->Amount Debited Order not confirmed', 
                      'Product Purchase Queries->Payment related Queries->NA', 
                      'Product Purchase Queries->Product Specific Information->Payment methods available', 
                      'Order processing Related->Online Payment Issues->EGV not Getting Applied', 
                      'Product Purchase Queries->Product Specific Information->How to use/redeem EGV',
                      'Payments / Wallet->e-Gift Voucher->e-Gift Voucher/Pin Not received',
                     'Order processing Related->Online Payment Issues->Issue with COD to prepaid conversion',
                     'Product Purchase Queries->Product Specific Information->superPay Related Queries',
                     'Product Purchase Queries->Product Specific Information->superPay Related Issues',
                     'Product Purchase Queries->Product Specific Information->Enquiry about Flipkart EMI - Credit for All']
filtered_data11["ssi"] = pd.Categorical(filtered_data11["ssi"], categories=transaction_order7, ordered = True)
                                                    
Adonc_SSI_summary_source = (
        filtered_data11.groupby(["session_dt","ssi"], observed= False)["contacts"].sum().reset_index()
        .rename(columns={"contacts":"Adonc_ssi_counts1"})
)
pivot_source_contacts = Adonc_SSI_summary_source.pivot(index = "ssi", 
                                                columns = "session_dt",
                                                values = "Adonc_ssi_counts1").fillna(0)

pivot_source_contacts.columns=pd.to_datetime(pivot_source_contacts.columns, errors='coerce')
pivot_source_contacts = pivot_source_contacts.sort_index(axis=1)
pivot_source_contacts.columns = pivot_source_contacts.columns.strftime('%d-%m-%Y')
pivot_source_contacts.reset_index(inplace=True)

for col in pivot_source_contacts.columns[1:]:
    if col in conversion_map:
        pivot_source_contacts[col] = pivot_source_contacts[col] * conversion_map[col]


pivot_source_contacts.iloc[:, 1:] = pivot_source_contacts.iloc[:, 1:].fillna(0).round(0).astype(int)

###-----3_P___1_P--------------------------##

##-------------------------------------##
filtered_data2= merged_df[(merged_df["Minutes_flag"].isin(["Minutes"])) &
                          (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
                          (merged_df["JN"]=="PAYMENTS")].copy()

emi_pg_ids =['cfa_juspay_dmi_emi',
    'cfa_juspay_fibe_emi',
    'cfa_juspay_abfl_emi',
    'cfa_juspay_chola_emi',
    'cfa_juspay_tvs_emi'
]

filtered_data2["payment_instrument"] = filtered_data2.apply(
lambda row: 'fk_emi' if row['pg_id'] in emi_pg_ids else row['payment_instrument'], axis=1)

filtered_data2["payment_instrument"] = filtered_data2["payment_instrument"].replace({"EGV":"EGV_wallet","EGV_WALLET":"EGV_wallet"})
transaction_order13 = ["EMI","fk_emi"]

filtered_data2["payment_instrument"] =pd.Categorical(filtered_data2["payment_instrument"], categories=transaction_order13, ordered=True)

payment_instrument_summary10 = (filtered_data2.groupby(["payment_instrument","session_dt","emi_flag"], observed=False)["contacts"].sum().reset_index()
                              .rename(columns={"contacts": "payment_instrument_count10"}))

pivot_payment_instrument_countact10 = payment_instrument_summary10.pivot_table(index= ["payment_instrument","emi_flag"],
                                                                     columns="session_dt",
                                                                     values = "payment_instrument_count10",
                                                                     aggfunc ='sum',
                                                                     observed =False).fillna(0)

pivot_payment_instrument_countact10.columns=pd.to_datetime(pivot_payment_instrument_countact10.columns, errors='coerce')
pivot_payment_instrument_countact10 = pivot_payment_instrument_countact10.sort_index(axis=1)
pivot_payment_instrument_countact10.columns = pivot_payment_instrument_countact10.columns.strftime('%d-%m-%Y')
pivot_payment_instrument_countact10.reset_index(inplace=True)

for col in pivot_payment_instrument_countact10.columns[2:]:
    if col in conversion_map:
        pivot_payment_instrument_countact10[col] = pivot_payment_instrument_countact10[col] * conversion_map[col]

# Final rounding
pivot_payment_instrument_countact10.iloc[:, 2:] = pivot_payment_instrument_countact10.iloc[:, 2:].fillna(0).round(0).astype(int)


#print(pivot_payment_instrument_countact10)

##-------------------------------------##
#pivot_payment_Adonc_instrument_countact###

filtered_data13= merged_df[(merged_df["Minutes_flag"].isin(["Minutes"])) &
                          (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
                          (merged_df["JN"]=="PAYMENTS") &
                          (merged_df["ssi"]=="Order processing Related->Online Payment Issues->Amount Debited Order not confirmed")].copy()

emi_pg_ids =['cfa_juspay_dmi_emi',
             'cfa_juspay_fibe_emi',
             'cfa_juspay_abfl_emi',
             'cfa_juspay_chola_emi',
             'cfa_juspay_tvs_emi'
]

filtered_data13["payment_instrument"] = filtered_data13.apply(
lambda row: 'fk_emi' if row['pg_id'] in emi_pg_ids else row['payment_instrument'], axis=1)

filtered_data13["payment_instrument"] = filtered_data13["payment_instrument"].replace({"EGV":"EGV_wallet","EGV_WALLET":"EGV_wallet"})

transaction_order14 = ["EMI","fk_emi"]
filtered_data13["payment_instrument"] =pd.Categorical(filtered_data13["payment_instrument"], categories=transaction_order14, ordered=True)

payment_instrument_Adonc_summary11 = (filtered_data13.groupby(["payment_instrument","session_dt","emi_flag"], observed=False)["contacts"].sum().reset_index()
                              .rename(columns={"contacts": "payment_instrument_Adonc_count1"}))

pivot_payment_instrument_Adonc_countact11 = payment_instrument_Adonc_summary11.pivot_table(index=["payment_instrument","emi_flag"],
                                                                     columns="session_dt",
                                                                     values = "payment_instrument_Adonc_count1",
                                                                     aggfunc ='sum',
                                                                     observed =False).fillna(0)

pivot_payment_instrument_Adonc_countact11.columns=pd.to_datetime(pivot_payment_instrument_Adonc_countact11.columns, errors='coerce')
pivot_payment_instrument_Adonc_countact11 = pivot_payment_instrument_Adonc_countact11.sort_index(axis=1)
pivot_payment_instrument_Adonc_countact11.columns = pivot_payment_instrument_Adonc_countact11.columns.strftime('%d-%m-%Y')
pivot_payment_instrument_Adonc_countact11.reset_index(inplace=True)

for col in pivot_payment_instrument_Adonc_countact11.columns[2:]:
    if col in conversion_map:
        pivot_payment_instrument_Adonc_countact11[col] = pivot_payment_instrument_Adonc_countact11[col] * conversion_map[col]

# Final rounding
pivot_payment_instrument_Adonc_countact11.iloc[:, 2:] = pivot_payment_instrument_Adonc_countact11.iloc[:, 2:].fillna(0).round(0).astype(int)



##------------overall summary-------------

filtered_data8= merged_df[(merged_df["Minutes_flag"].isin(["Minutes"])) &
                          (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"]))  
                           ].copy()


Overall_refund_summary = (filtered_data8.groupby(["session_dt"], observed=False)["contacts"].sum().reset_index()
                              .rename(columns={"contacts": "payment_overall_count"}))

pivot_Overall_refund_summary_countact = Overall_refund_summary.pivot_table(index=None,
                                                                     columns="session_dt",
                                                                     values = "payment_overall_count",
                                                                     aggfunc ='sum',
                                                                     observed =False).fillna(0)

pivot_Overall_refund_summary_countact.columns=pd.to_datetime(pivot_Overall_refund_summary_countact.columns, errors='coerce')
pivot_Overall_refund_summary_countact = pivot_Overall_refund_summary_countact.sort_index(axis=1)
pivot_Overall_refund_summary_countact.columns = pivot_Overall_refund_summary_countact.columns.strftime('%d-%m-%Y')
pivot_Overall_refund_summary_countact.reset_index(inplace=True)

for col in pivot_Overall_refund_summary_countact.columns[1:]:
    if col in conversion_map:
        pivot_Overall_refund_summary_countact[col] = pivot_Overall_refund_summary_countact[col] * conversion_map[col]

# Final rounding
pivot_Overall_refund_summary_countact.iloc[:, 1:] = pivot_Overall_refund_summary_countact.iloc[:, 1:].fillna(0).round(0).astype(int)

###--pg_id_Payment related Queries->NA------------


filtered_data14= merged_df[(merged_df["Minutes_flag"].isin(["Minutes"])) &
                          (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
                          (merged_df["JN"]=="PAYMENTS") &
                          (merged_df["ssi"]=="Order processing Related->Online Payment Issues->Amount Debited Order not confirmed")].copy()

filtered_data14['pg_id']=filtered_data14['pg_id'].astype(str).str.lower()

pg_ids_units =[ 
    'payu_intent_hl',
'lockin',
'payu_3ds_cc_hl',
'paytm_upi',
'paytm_3ds_cc_hl',
'payu_collect_hl',
'qc',
'juspay_fk_upi',
'payu_3ds_dc_hl',
'tpsl_pg_category_b'
]

filtered_data14['pg_id'] = filtered_data14['pg_id'].apply(
   lambda x: x.lower() if x.lower() in [pg.lower() for pg in pg_ids_units] else 'Others' )

transaction_order8 = [ 'payu_intent_hl',
'lockin',
'payu_3ds_cc_hl',
'paytm_upi',
'paytm_3ds_cc_hl',
'payu_collect_hl',
'qc',
'juspay_fk_upi',
'payu_3ds_dc_hl',
'tpsl_pg_category_b']

filtered_data14["pg_id"] =pd.Categorical(filtered_data14["pg_id"], categories=transaction_order8, ordered=True)

payment_pg_summary1 = (filtered_data14.groupby(["pg_id","session_dt"], observed=False)["contacts"].sum().reset_index()
                              .rename(columns={"contacts": "pg_id_count"}))

pivot_payment_pg_summary1 = payment_pg_summary1.pivot_table(index="pg_id",
                                                                     columns="session_dt",
                                                                     values = "pg_id_count",
                                                                     aggfunc ='sum',
                                                                     observed =False).fillna(0)

pivot_payment_pg_summary1.columns=pd.to_datetime(pivot_payment_pg_summary1.columns, errors='coerce')
pivot_payment_pg_summary1 = pivot_payment_pg_summary1.sort_index(axis=1)
pivot_payment_pg_summary1.columns = pivot_payment_pg_summary1.columns.strftime('%d-%m-%Y')
pivot_payment_pg_summary1.reset_index(inplace=True)

for col in pivot_payment_pg_summary1.columns[1:]:
    if col in conversion_map:
        pivot_payment_pg_summary1[col] = pivot_payment_pg_summary1[col] * conversion_map[col]

# Final rounding
pivot_payment_pg_summary1.iloc[:, 1:] = pivot_payment_pg_summary1.iloc[:, 1:].fillna(0).round(0).astype(int)


#print(pivot_payment_pg_summary1)

###--pg_id_Amount Debited Order not confirmed------------


filtered_data15= merged_df[(merged_df["Minutes_flag"].isin(["Minutes"])) &
                          (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
                          (merged_df["JN"]=="PAYMENTS") &
                          (merged_df["ssi"]=="Product Purchase Queries->Payment related Queries->NA")].copy()

filtered_data15['pg_id']=filtered_data15['pg_id'].astype(str).str.lower()

pg_ids_units =[ 
    'payu_intent_hl',
'lockin',
'payu_3ds_cc_hl',
'paytm_upi',
'paytm_3ds_cc_hl',
'payu_collect_hl',
'qc',
'juspay_fk_upi',
'payu_3ds_dc_hl',
'tpsl_pg_category_b'

]

filtered_data15['pg_id'] = filtered_data15['pg_id'].apply(
   lambda x: x.lower() if x.lower() in [pg.lower() for pg in pg_ids_units] else 'Others' )

transaction_order8 = [ 'payu_intent_hl',
'lockin',
'payu_3ds_cc_hl',
'paytm_upi',
'paytm_3ds_cc_hl',
'payu_collect_hl',
'qc',
'juspay_fk_upi',
'payu_3ds_dc_hl',
'tpsl_pg_category_b']

filtered_data15["pg_id"] =pd.Categorical(filtered_data15["pg_id"], categories=transaction_order8, ordered=True)

payment_pg_summary2 = (filtered_data15.groupby(["pg_id","session_dt"], observed=False)["contacts"].sum().reset_index()
                              .rename(columns={"contacts": "pg_id_count1"}))

pivot_payment_pg_summary2 = payment_pg_summary2.pivot_table(index="pg_id",
                                                                     columns="session_dt",
                                                                     values = "pg_id_count1",
                                                                     aggfunc ='sum',
                                                                     observed =False).fillna(0)

pivot_payment_pg_summary2.columns=pd.to_datetime(pivot_payment_pg_summary2.columns, errors='coerce')
pivot_payment_pg_summary2 = pivot_payment_pg_summary2.sort_index(axis=1)
pivot_payment_pg_summary2.columns = pivot_payment_pg_summary2.columns.strftime('%d-%m-%Y')
pivot_payment_pg_summary2.reset_index(inplace=True)

for col in pivot_payment_pg_summary2.columns[1:]:
    if col in conversion_map:
        pivot_payment_pg_summary2[col] = pivot_payment_pg_summary2[col] * conversion_map[col]

# Final rounding
pivot_payment_pg_summary2.iloc[:, 1:] = pivot_payment_pg_summary2.iloc[:, 1:].fillna(0).round(0).astype(int)


#print(pivot_payment_pg_summary2)


output_path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Hyperlocal_transaction_output.xlsx"
with pd.ExcelWriter(output_path, engine = 'openpyxl') as writer:
    final_summary.to_excel(writer, sheet_name ='Overall_summary_view', index=False)
    final_summary1.to_excel(writer, sheet_name ='payment_reason_Adonc_countact', index=False )
    pivot_transaction_source_countact.to_excel(writer, sheet_name ='Payment_Tranaction_Source_count', index=False )
    pivot_payment_instrument_countact.to_excel(writer, sheet_name = 'pivot_payment_instrument_countact', index=False)
    pivot_payment_instrument_Adonc_countact.to_excel(writer, sheet_name = 'pivot_payment_instrument_Adonc_countact', index=False)
    pivot_source_contacts.to_excel(writer, sheet_name = 'pivot_SSI_counts', index=False)
    pivot_Overall_refund_summary_countact.to_excel(writer, sheet_name = 'Overall_count', index=False)
    pivot_payment_instrument_countact10.to_excel(writer, sheet_name = 'Overall_EMI_3_P', index=False)
    pivot_payment_instrument_Adonc_countact11.to_excel(writer, sheet_name = 'Adonc_EMI_3_p', index=False)
    pivot_payment_pg_summary1.to_excel(writer, sheet_name = 'Adonc_pg_id', index=False)
    pivot_payment_pg_summary2.to_excel(writer, sheet_name = 'Payment_related_pg_id', index=False)
    
   
print("Excel file successfully saved:",output_path)


datetime64[ns]
2025-12-01 00:00:00
Excel file successfully saved: G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Hyperlocal_transaction_output.xlsx


In [20]:
print(pivot_source_contacts)

session_dt                                                ssi  01-01-2025  \
0           Order processing Related->Online Payment Issue...      3947.0   
1           Product Purchase Queries->Payment related Quer...      1063.0   
2           Product Purchase Queries->Product Specific Inf...      1229.0   
3           Order processing Related->Online Payment Issue...       593.0   
4           Product Purchase Queries->Product Specific Inf...       429.0   
5           Payments / Wallet->e-Gift Voucher->e-Gift Vouc...       274.0   
6           Order processing Related->Online Payment Issue...         1.0   
7           Product Purchase Queries->Product Specific Inf...         0.0   
8           Product Purchase Queries->Product Specific Inf...         0.0   

session_dt  02-01-2025  03-01-2025  04-01-2025  05-01-2025  06-01-2025  \
0               4288.0      4246.0      4260.0      4534.0      3702.0   
1               1197.0      1202.0      1272.0      1221.0      1161.0   
2      

In [24]:
filtered_data17= merged_df[(merged_df["Minutes_flag"].isin(["Minutes"])) &
                          (~merged_df["issue_type"].isin(["CABN", "Mobile Recharge & Bill Payments"])) &
                          (merged_df["JN"]=="PAYMENTS") &
                          (merged_df["ssi"]=="Product Purchase Queries->Payment related Queries->NA")].copy()

# Step 2: Normalize pg_id to lowercase
filtered_data17['pg_id'] = filtered_data17['pg_id'].astype(str).str.lower()

# Step 3: Define PG ID groups (all lowercase)
pg_id_groups = {
    'UPI_INTENT': ['juspay_intent', 'payu_intent', 'payu_intent_hl', 'paytm_upi', 'payu_gpay'],
    'CREDIT': ['paytm_3ds_cc', 'payu_auth_3ds_sbt', 'cybs_hdfcauth', 'payu_3ds', 'payu_current_egv'],
    'EMI': ['razorpay_3ds_emi_bob', 'idfc_advanz_efa', 'paytm_3ds_emi', 'bajaj_3ds_finserv_pg', 'cfa_flexmoney_emi'],
    'EGV_WALLET': ['qc'],
    'DEBIT': ['paytm_3ds_dc', 'razorpay_3ds', 'payu_3ds', 'payu_3ds_dc_sbi', 'payu_3ds_dc_hl']
}

# Step 4: Filter valid PG IDs
valid_pg_ids = [pg for pg_list in pg_id_groups.values() for pg in pg_list]
filtered_data17['pg_id'] = filtered_data17['pg_id'].apply(lambda x: x if x in valid_pg_ids else 'Others')
filtered_data17 = filtered_data17[filtered_data17['pg_id'] != 'Others']

# Step 5: Filter selected instruments
selected_instruments = list(pg_id_groups.keys())
filtered_data17 = filtered_data17[filtered_data17['payment_instrument'].isin(selected_instruments)]
filtered_data17["payment_instrument"] = pd.Categorical(filtered_data17["payment_instrument"], categories=selected_instruments, ordered=True)

# Step 6: Apply PG ID ordering per instrument
for instrument in selected_instruments:
    pg_order = pg_id_groups[instrument]
    filtered_data17.loc[filtered_data17['payment_instrument'] == instrument, 'pg_id'] = pd.Categorical(
        filtered_data17.loc[filtered_data17['payment_instrument'] == instrument, 'pg_id'],
        categories=pg_order,
        ordered=True
    )

# Step 7: Group and pivot
Payment_instrument_pg_id_EGV = (
    filtered_data17.groupby(["session_dt", "payment_instrument", "pg_id"], observed=False)["contacts"]
    .sum()
    .reset_index()
    .rename(columns={"contacts": "pg_id_Payment_count"})
)

pivot_payment_instrument_PG_PI_summary = Payment_instrument_pg_id_EGV.pivot_table(
    index=["payment_instrument", "pg_id"],
    columns="session_dt",
    values="pg_id_Payment_count",
    aggfunc="sum",
    observed=False
).fillna(0)

pivot_payment_instrument_PG_PI_summary.columns = pd.to_datetime(pivot_payment_instrument_PG_PI_summary.columns, errors='coerce')
pivot_payment_instrument_PG_PI_summary = pivot_payment_instrument_PG_PI_summary.sort_index(axis=1)
pivot_payment_instrument_PG_PI_summary.columns = pivot_payment_instrument_PG_PI_summary.columns.strftime('%d-%m-%Y')
pivot_payment_instrument_PG_PI_summary.reset_index(inplace=True)

# Step 8: Sort PG IDs per instrument using categorical order
sorted_frames = []

for instrument in selected_instruments:
    pg_order = pg_id_groups[instrument]
    temp_df = pivot_payment_instrument_PG_PI_summary[
        pivot_payment_instrument_PG_PI_summary['payment_instrument'] == instrument
    ].copy()

    temp_df['pg_id'] = pd.Categorical(temp_df['pg_id'], categories=pg_order, ordered=True)
    temp_df['pg_id_code'] = temp_df['pg_id'].cat.codes
    temp_df.sort_values(by='pg_id_code', inplace=True)
    temp_df.drop(columns='pg_id_code', inplace=True)

    sorted_frames.append(temp_df)

pivot_payment_instrument_PG_PI_summary = pd.concat(sorted_frames, ignore_index=True)

# Step 9: Apply conversion map if needed
for col in pivot_payment_instrument_PG_PI_summary.columns[2:]:
    if col in conversion_map:
        pivot_payment_instrument_PG_PI_summary[col] = pivot_payment_instrument_PG_PI_summary[col] * conversion_map[col]

# Step 10: Final rounding and filtering
pivot_payment_instrument_PG_PI_summary.iloc[:, 2:] = (
    pivot_payment_instrument_PG_PI_summary.iloc[:, 2:].fillna(0).round(0).astype(int)
)

pivot_payment_instrument_PG_PI_summary = pivot_payment_instrument_PG_PI_summary[
    (pivot_payment_instrument_PG_PI_summary.drop(columns=['payment_instrument', 'pg_id']) != 0).any(axis=1)
]

# Step 11: Export to Excel
output_path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Payment_contacts_PG_Hyperlocal_Payment_ins.xlsx"
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    pivot_payment_instrument_PG_PI_summary.to_excel(writer, sheet_name='PG_Payment_ins', index=False)

print("Excel File saved Successfully:", output_path)


Excel File saved Successfully: G:\My Drive\New_Working_Files\KNIME\CTU_2025\Payment_Adonc\Payment_contacts_PG_Hyperlocal_Payment_ins.xlsx
